# arg-position-back-functions — faded example 3: Wire sub back fns into BackFuncs and dispatch

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `arg-position-back-functions`. Running the beacon reports progress on the `Backprop: Arg-position back funcs` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Arg-position back funcs` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`arg-position-back-functions`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "arg-position-back-functions"
DD_SUBTOPIC = "Backprop: Arg-position back funcs"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The reverse pass dispatches by the key `(fwd_fn, argnum)`. You must register a back fn for every argnum the forward consumed; a lookup for an argnum you never registered raises `KeyError`. Here you complete the registration step for `out = x - y` so both positions resolve while a third position correctly fails.

## Faded exercise 3

### Complete the registration for `t.subtract`

`BackFuncs`, `sub_back0` (returns `grad_out`) and `sub_back1` (returns `-grad_out`) are written for you, and `bf = BackFuncs()` is created. Complete the blanked step: register `sub_back0` at `(t.subtract, 0)` and `sub_back1` at `(t.subtract, 1)` so the dispatcher can look them up. Do not register argnum 2 — a lookup there must raise `KeyError`.

**Fill in:** the two add_back_func calls registering sub_back0 at (t.subtract, 0) and sub_back1 at (t.subtract, 1).

In [ ]:
class BackFuncs:
    def __init__(self):
        self._registry = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._registry[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._registry[(fwd_fn, argnum)]


def sub_back0(grad_out, out, x, y):
    return grad_out


def sub_back1(grad_out, out, x, y):
    return -grad_out


bf = BackFuncs()


def register(bf):
    raise NotImplementedError()  # TODO: register sub_back0 at (t.subtract, 0) and sub_back1 at (t.subtract, 1)


register(bf)

t.manual_seed(0)
x = t.randn(3, requires_grad=True)
y = t.randn(3, requires_grad=True)
out = x - y
grad_out = t.randn(3)
out.backward(grad_out)
gx = bf.get_back_func(t.subtract, 0)(grad_out, out.detach(), x.detach(), y.detach())
gy = bf.get_back_func(t.subtract, 1)(grad_out, out.detach(), x.detach(), y.detach())
print('grad_x match:', t.allclose(gx, x.grad))
print('grad_y match:', t.allclose(gy, y.grad))


def _test():
    t.manual_seed(0)
    x = t.randn(3, requires_grad=True)
    y = t.randn(3, requires_grad=True)
    out = x - y
    grad_out = t.randn(3)
    out.backward(grad_out)
    f0 = bf.get_back_func(t.subtract, 0)
    f1 = bf.get_back_func(t.subtract, 1)
    gx = f0(grad_out, out.detach(), x.detach(), y.detach())
    gy = f1(grad_out, out.detach(), x.detach(), y.detach())
    assert t.allclose(gx, x.grad)
    assert t.allclose(gy, y.grad)
    raised = False
    try:
        bf.get_back_func(t.subtract, 2)
    except KeyError:
        raised = True
    assert raised, 'unregistered argnum must raise KeyError'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class BackFuncs:
    def __init__(self):
        self._registry = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._registry[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._registry[(fwd_fn, argnum)]


def sub_back0(grad_out, out, x, y):
    return grad_out


def sub_back1(grad_out, out, x, y):
    return -grad_out


bf = BackFuncs()


def register(bf):
    bf.add_back_func(t.subtract, 0, sub_back0)
    bf.add_back_func(t.subtract, 1, sub_back1)


register(bf)

t.manual_seed(0)
x = t.randn(3, requires_grad=True)
y = t.randn(3, requires_grad=True)
out = x - y
grad_out = t.randn(3)
out.backward(grad_out)
gx = bf.get_back_func(t.subtract, 0)(grad_out, out.detach(), x.detach(), y.detach())
gy = bf.get_back_func(t.subtract, 1)(grad_out, out.detach(), x.detach(), y.detach())
print('grad_x match:', t.allclose(gx, x.grad))
print('grad_y match:', t.allclose(gy, y.grad))
```
</details>